# Error analysis

This notebook is for week 9, step 3. It shows the traces of one run of `make testset`, one at a time, and it saves your notes.

1. **Open coding.** Read each trace: the question, the tool calls, the retrieved chunks and the answer. Decide if the output passes. For each failure, write a short note on the first thing that went wrong. Read at least 30 traces.
2. **Axial coding.** Read all your notes again, and put each failure into a category. Then count the categories.

The notes go to `evals/error_notes.jsonl`, and the table of categories goes to `evals/failure_categories.csv`. The checks of week 9 read both files, so commit them. Langfuse must run, because the notebook reads each trace from it.

In [ ]:
import csv
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

# Find the repository root, so that `evals` can be imported.
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from langfuse import get_client

langfuse = get_client()
NOTES = ROOT / "evals" / "error_notes.jsonl"
TABLE = ROOT / "evals" / "failure_categories.csv"

runs = sorted((ROOT / "evals" / "results").glob("testset-*.jsonl"))
assert runs, "No result file. Run `make testset` first."
RUN = runs[-1]  # The last run. Put another path here to read an older run.
rows = [json.loads(line) for line in RUN.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{RUN.name}: {len(rows)} cases, code_pass {sum(row['code_pass'] for row in rows)}")

## Show one trace

`show(n)` prints case number `n` of the run. The tool calls and the retrieved chunks come from the trace in Langfuse. `code_pass` is only a hint: read the output and decide yourself.

In [ ]:
def short(text, size=300):
    text = text if isinstance(text, str) else json.dumps(text, ensure_ascii=False)
    return text if len(text) <= size else text[:size] + " ..."


def observations(trace_id):
    page = langfuse.api.observations.get_many(trace_id=trace_id, fields="core,basic,io", limit=1000)
    return sorted(page.data, key=lambda o: o.start_time)


def show(n):
    row = rows[n]
    print(f"[{n}] {row['id']} ({row['language']})  code_pass={row['code_pass']}  trace {row['trace_id']}")
    print("QUESTION:", row["question"])
    print("EXPECTED:", row["expected_dataset_ids"] or "", row["expected_chunk_ids"] or "")
    for o in observations(row["trace_id"]):
        if o.type == "TOOL":
            print(f"\nTOOL {o.name} {'ERROR ' if o.level == 'ERROR' else ''}input={short(o.input, 200)}")
            print("  output:", short(o.output or ""))
        elif o.type == "RETRIEVER":
            print(f"  RETRIEVER {o.name}: {short(o.output or '', 400)}")
    print("\nANSWER:", row["answer"])
    print("DATASET IDS:", row["dataset_ids"])
    for passage in row["cited_passages"]:
        print(f"CITED {passage['chunk_id']}: {short(passage['text'], 200)}")
    if row["errors"]:
        print("ERRORS:", row["errors"])


show(0)

## Open coding: one note for each trace that you read

`note(n, passed, text)` saves your decision for case number `n`. A failure needs a note: what went wrong first, in your own words (for example "searched for 'water' only and missed the desalination dataset"). A second call for the same case replaces the note.

In [ ]:
def load_notes():
    return [json.loads(line) for line in NOTES.read_text(encoding="utf-8").splitlines() if line.strip()] if NOTES.exists() else []


def save_notes(notes):
    NOTES.write_text("".join(json.dumps(note, ensure_ascii=False) + "\n" for note in notes), encoding="utf-8")


def note(n, passed, text=""):
    row = rows[n]
    if not passed and len(text.split()) < 3:
        raise ValueError("Write a short note on the failure: at least three words.")
    notes = [item for item in load_notes() if item["case_id"] != row["id"]]
    old = next((item for item in load_notes() if item["case_id"] == row["id"]), {})
    notes.append({
        "case_id": row["id"],
        "trace_id": row["trace_id"],
        "passed": bool(passed),
        "note": text,
        "category": old.get("category", ""),
    })
    save_notes(notes)
    print(f"{len(notes)} notes, {sum(not item['passed'] for item in notes)} failures")


# Example (remove it, and write your own decisions):
# note(0, True)
# note(1, False, "cites a chunk of the right report, but the page does not hold the sample size")

## Axial coding: put each failure into a category

Read all failure notes, and find the groups. Give each group a short name and a description. Then set the category of each failure in `CATEGORIES`. Two to eight categories are usual. A category that only you understand is not useful: the name must say what fails.

In [ ]:
failures = [item for item in load_notes() if not item["passed"]]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(failures, columns=["case_id", "note", "category"])

In [ ]:
# The description of each category.
DESCRIPTIONS = {
    # "wrong dataset chosen": "The search returned the correct dataset, but the answer names another one.",
}

# The category of each failure: case ID -> category name.
CATEGORIES = {
    # "t017": "wrong dataset chosen",
}

In [ ]:
notes = load_notes()
for item in notes:
    if not item["passed"]:
        item["category"] = CATEGORIES.get(item["case_id"], item.get("category", ""))
save_notes(notes)

failures = [item for item in notes if not item["passed"]]
missing = [item["case_id"] for item in failures if not item["category"]]
unknown = sorted({item["category"] for item in failures if item["category"]} - set(DESCRIPTIONS))
counts = Counter(item["category"] for item in failures if item["category"])
with TABLE.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["category", "count", "description"])
    for category, count in counts.most_common():
        writer.writerow([category, count, DESCRIPTIONS.get(category, "")])
print(f"{len(notes)} traces read, {len(failures)} failures. Failures without a category: {missing}")
print(f"Categories without a description: {unknown}")
pd.read_csv(TABLE) if counts else None